In [11]:
chr(0)

'\x00'

In [12]:
repr('\x00')

"'\\x00'"

In [13]:
print('\x00')

 


In [14]:
chr(0)

'\x00'

In [15]:
print(chr(0))

 


In [16]:
"this is a test" + chr(0) + "string"

'this is a test\x00string'

In [17]:
print("this is a test" + chr(0) + "string")

this is a test string


In [18]:
test_string = "hello! こんにちは!"
utf8_encoded = test_string.encode("utf-8")
print(utf8_encoded)
print(type(utf8_encoded))
list(utf8_encoded)

b'hello! \xe3\x81\x93\xe3\x82\x93\xe3\x81\xab\xe3\x81\xa1\xe3\x81\xaf!'
<class 'bytes'>


[104,
 101,
 108,
 108,
 111,
 33,
 32,
 227,
 129,
 147,
 227,
 130,
 147,
 227,
 129,
 171,
 227,
 129,
 161,
 227,
 129,
 175,
 33]

In [19]:
print(len(test_string))
print(len(utf8_encoded))
print(utf8_encoded.decode("utf-8"))

13
23
hello! こんにちは!


7. re.finditer、merge 规则与 tie-break

In [22]:
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

In [24]:
import regex as re
re.findall(PAT, "some text that i'll pre-tokenize")

['some', ' text', ' that', ' i', "'ll", ' pre', '-', 'tokenize']

完了，num_processes=8 明显快很多，已经低于 2 分钟。

结果：

vocab_len: 10000
merges_len: 9743
墙钟时间：1:28.66
Python 内部计时：88.41s
CPU 使用：628%
峰值内存：1548844 KB，约 1.48 GiB
最长 token：b' accomplishment'
最长 token 长度：15 bytes
summary 保存到了：

artifacts/tinystories_bpe_increase_10000_8proc_summary.json

对比之前 4 进程：

4 进程：2:27.05
8 进程：1:28.66
所以 hint 里的 “低于 2 分钟” 现在达到了。8 进程这版可以作为你 TinyStories BPE 训练结果来写。

Problem (learning_rate): Tune the learning rate 调整学习率 (2 B200 hrs; 3
points)

判断标准：

如果 1e-3 更快下降且稳定，说明还没到边界。
如果 1e-3 开始震荡但不炸，它可能接近 edge。
如果 3e-3 loss 上升、剧烈震荡或 NaN，那它就是 divergent run。
如果 3e-3 也没炸，再加一个 1e-2。

In [ ]:
在 low-resource 设置下，我比较了 2e-3 和 3e-3。两者都稳定收敛，最终 validation loss 都约在 1.67-1.69 附近。2e-3 在本次实验中取得了略低的 validation loss，因此我将其视为更稳健的选择；不过二者差距较小，可能受到 validation batch 采样噪声影响。

Problem (batch_size_experiment): Batch size variations Batch size 变化 (1
B200 hr; 1 point)

In [ ]:
在固定训练步数为 500 steps、学习率为 3e-3 的设置下，batch size 越大，每一步处理的 token 数越多，因此总训练 token 数也越多，最终 validation loss 明显更低。实验中，batch size 从 1 增加到 256 时，validation loss 从约 4.20 降低到约 1.80，perplexity 也从约 66.5 降低到约 6.1。说明在这个设置下，更大的 batch size 带来了更稳定、更有效的训练。

不过你也要加一句限制：

但这个比较并不是完全公平的，因为不同 batch size 在相同 step 数下处理的 token 总量不同。batch size 256 在 500 steps 中处理了约 3277 万 tokens，而 batch size 1 只处理了 12.8 万 tokens。因此，分析 learning curves 时更应该使用 tokens_processed 作为横轴，而不是只使用 step。

再加一句和作业要求相关的：

本实验覆盖了 batch size 1、4、16、32、64、128、256，其中包含作业要求中特别提到的 64 和 128。所有这些 batch size 都可以在当前 GPU 上运行，尚未达到显存上限。

存在一个“最大有用 batch size”附近的临界区域；超过后，继续增大 batch 对 sample efficiency 的帮助会变小，主要只是提高并行吞吐
但超过某个临界区域后：继续增大 batch，不会显著减少达到同样 loss 所需的 token 数，而不是样本效率

所以大 batch 常常需要配合 提高学习率 或 增加训练步数。


Batch size 变大：梯度更准，但噪声更小，梯度估计方差越小，训练曲线通常更平滑。但噪声不是纯坏事。小 batch 的噪声有时类似正则化，可以帮助模型跳出尖锐极小值


batch size 变大后：

提高 learning rate
或增加 training steps
或调整 warmup / decay schedule

Problem (generate): Generate text 生成文本 (1 point)

In [ ]:
设最大200上下文
Once upon a time, there was a little boy named Tim. He had a dog named Max. Tim and Max liked to play together. One day, Tim was very hungry. He wanted to find some food.
Tim and Max walked around the park. They saw a big tree. Max said, "Let's go to the tree and look for food!" Tim agreed, and they went to the tree.
Tim and Max found a big, red apple. They took a bite and it was very tasty. Tim and Max were very happy. They ate the apple and had a great day.
<|endoftext|>


这个样本整体质量较好。模型生成了一个完整的小故事，包含人物、目标和结尾，并且自动生成了 <|endoftext|> 作为文档结束符。文本语法基本正确，符合 TinyStories 的简单儿童故事风格。缺点是人物名和短语有一定重复，情节也比较简单，但作为 low-resource 训练结果已经较好。

Problem (layer_norm_ablation): Remove RMSNorm and train 移除 RMSNorm 并训练 (0.5 B200 hrs; 1 point)



In [ ]:
先跑了500步32batch的对比实验
：移除 RMSNorm 后，模型对学习率明显更敏感。
no-RMSNorm lr	step 500 val loss	状态
3e-4	2.8720	稳定但慢
1e-3	2.4507	稳定
2e-3	2.3791	短跑最好
3e-3	崩了	train_loss 到 23983，val perplexity 溢出

In [ ]:
no-RMSNorm, lr=2e-3, 5000 steps  step 1000 开始 NaN

1e-3	5000	1.6730	稳定


因此，RMSNorm 的主要作用是提升优化稳定性，并允许使用更大的 learning rate。
最终 loss 在本实验中差距不大，但无 RMSNorm 对 learning rate 更敏感。


| 架构                       |  对学习率的敏感性 | 常见现象                              |
| ------------------------ | --------: | --------------------------------- |
| **Pre-RMSNorm**          |        较低 | 更稳定，可用较大学习率，warmup 要求较弱           |
| **Post-RMSNorm**         |        较高 | 大学习率更容易 loss spike/发散，通常需要 warmup |
| **无 Norm 或 Norm 很弱**     |        很高 | 深层模型训练困难，学习率窗口窄                   |
| **RMSNorm 替代 LayerNorm** | 略更省算、尺度更稳 | 学习率可稍微更宽容，但不是万能                   |


Problem (pre_norm_ablation): Implement post-norm and train 实现 postnorm 并训练 (0.5 B200 hrs; 1 point)

In [ ]:
Post-norm 在 500-step sweep 中没有像 no-RMSNorm 那样发散，说明保留 RMSNorm 仍然提供了稳定性。
但 post-norm 的最佳短跑 validation loss 约为 2.31，比 pre-norm baseline 同等 token 数下通常更差，后续需要完整 5000-step 曲线比较最终收敛效果。

In [ ]:
Post-norm 在 lr=3e-3 下可以稳定训练，没有像 no-RMSNorm 那样发散。但相比 pre-norm baseline，post-norm 的 validation loss 略高：pre-norm 最终 val_loss=1.6931，post-norm 最终 val_loss=1.7202。
这说明在本实验设置下，post-norm 并不会立即导致训练失败，但优化效果略差。Pre-norm 的 residual stream 更直接，可能带来更好的梯度流动和收敛表现。

在 post-norm ablation 中，所有测试学习率都能完成训练，没有出现 no-RMSNorm 那样的 NaN 发散。但学习率对最终结果影响明显：虽然 3e-3 在 500-step 短跑中下降最快，但完整 5000-step 训练中 1e-3 得到最低的 validation loss。最佳 post-norm run 的 final validation loss 为 1.6939，与 pre-norm baseline 的 1.6931 非常接近。不过 pre-norm 可以在 3e-3 下稳定训练并取得较好结果，而 post-norm 的最佳学习率更小，说明 post-norm 对 learning rate 更敏感一些。

跑完了，post-norm 四个完整 5000-step 学习率结果如下：

post-norm lr	final val loss	best val loss	是否发散
3e-4	1.8376	1.8142	否
1e-3	1.6939	1.6523	否
2e-3	1.7012	1.6899	否
3e-3	1.7202	1.7158	否

Problem (no_pos_emb): Implement NoPE 实现 NoPE (0.5 B200 hrs; 1 point)

NoPE LR	step 500 val_loss	ppl
3e-4	3.3233	27.75
1e-3	2.8133	16.66
2e-3	2.6700	14.44
3e-3	2.7180	15.15

设置	LR	final val_loss	best val_loss	best step
NoPE	2e-3	1.7496	1.7326	4800
RoPE 同 LR	2e-3	1.6698	1.6238	4800
RoPE 之前 baseline	3e-3	1.6931	1.6639	4900


说明位置编码对语言建模有帮助
